<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-2-data-pipelines/Module_2_Session_2_PDF_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Session 2: PDF Data Pipelines

## What we are building
Swiggy has hundreds of internal documents — delivery partner policies,
restaurant contracts, food safety guidelines — all stored as PDFs.
Before an LLM can answer questions from these documents, we need a pipeline that:
1. Extracts text from the PDF
2. Splits it into manageable chunks
3. Prepares those chunks for LLM input

## Real-world equivalent
At Swiggy, this would be the first step before building a RAG system on AWS Bedrock
using Bedrock Knowledge Bases + Amazon OpenSearch Serverless.

## Step 1 — Install and Import Libraries
New library this session: `PyMuPDF` (imported as `fitz`)
- PyMuPDF is the most popular Python library for reading PDF files
- It lets us open a PDF and extract text page by page
- `fitz` is just the internal name PyMuPDF uses — don't let it confuse you

We also use `groq` and `os` which you already know.

In [1]:
# Install libraries
# PyMuPDF is new this session — it lets us read PDF files
!pip install pymupdf groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.3 MB/s eta 0:00:00


In [2]:
# Import libraries
import fitz  # PyMuPDF — fitz is the internal name, used to open and read PDFs
from groq import Groq  # to call the LLM
import os  # to read environment variables
from google.colab import userdata  # to safely read Colab Secrets

## Step 2 — Create a Sample PDF
We create a simple Swiggy Delivery Partner Policy PDF.
In real life this would already exist in S3 or Google Drive.

In [3]:
# We will create a simple PDF using PyMuPDF itself
# This simulates a Swiggy internal policy document

# Create a new PDF document
doc = fitz.open()  # opens a blank PDF in memory

# Add a new page
page = doc.new_page()

# The text we want to put in our PDF
policy_text = """
SWIGGY DELIVERY PARTNER POLICY — 2024

1. WORKING HOURS
Delivery partners can choose their own working hours.
Minimum commitment is 4 hours per day to remain active on the platform.
Partners working more than 8 hours get a productivity bonus of ₹50 per day.

2. PAYMENT POLICY
Base pay is ₹25 per delivery.
Long distance deliveries above 5km get an extra ₹10.
Payments are processed every Monday by 10am.

3. CANCELLATION POLICY
Partners can cancel up to 2 orders per day without penalty.
More than 2 cancellations results in a ₹20 deduction per cancellation.
Repeated cancellations may lead to account suspension.

4. RATINGS POLICY
Partners must maintain a minimum rating of 3.5 out of 5.
Partners below 3.5 rating for 2 consecutive weeks enter a performance review.
Partners above 4.5 rating for a month receive a bonus of ₹500.

5. SAFETY POLICY
Helmets are mandatory at all times during delivery.
Violation of safety rules leads to immediate suspension for 7 days.
Three violations result in permanent deactivation from the platform.
"""

# Insert the text into the page
page.insert_text((50, 50), policy_text, fontsize=11)

# Save the PDF to disk
doc.save("swiggy_policy.pdf")
doc.close()

print("PDF created successfully!")

PDF created successfully!


## Step 3 — Extract Text from the PDF
Now we open the PDF we just created and extract text page by page.
This is the core skill — in real life the PDF would come from S3 or Google Drive.

In [4]:
# Open the PDF file
doc = fitz.open("swiggy_policy.pdf")

# How many pages does it have?
print(f"Total pages: {len(doc)}")

# Loop through each page and extract text
for page_num in range(len(doc)):
    page = doc[page_num]  # get the page by index
    text = page.get_text()  # extract all text from that page
    print(f"\n--- Page {page_num + 1} ---")
    print(text)

doc.close()

Total pages: 1

--- Page 1 ---
SWIGGY DELIVERY PARTNER POLICY · 2024
1. WORKING HOURS
Delivery partners can choose their own working hours.
Minimum commitment is 4 hours per day to remain active on the platform.
Partners working more than 8 hours get a productivity bonus of ·50 per day.
2. PAYMENT POLICY
Base pay is ·25 per delivery.
Long distance deliveries above 5km get an extra ·10.
Payments are processed every Monday by 10am.
3. CANCELLATION POLICY
Partners can cancel up to 2 orders per day without penalty.
More than 2 cancellations results in a ·20 deduction per cancellation.
Repeated cancellations may lead to account suspension.
4. RATINGS POLICY
Partners must maintain a minimum rating of 3.5 out of 5.
Partners below 3.5 rating for 2 consecutive weeks enter a performance review.
Partners above 4.5 rating for a month receive a bonus of ·500.
5. SAFETY POLICY
Helmets are mandatory at all times during delivery.
Violation of safety rules leads to immediate suspension for 7 days.
Thre

## Step 4 — Split Text into Chunks
We split the full text into smaller chunks — one per policy section.
In real life, chunking strategy is very important for RAG systems.
We will go deeper on this in Module 3 — RAG Pipelines.

In [5]:
# Extract full text from the PDF
doc = fitz.open("swiggy_policy.pdf")
full_text = doc[0].get_text()
doc.close()

# Split the text by numbered sections (1. 2. 3. etc.)
# We use a simple approach — split on newlines and group by section
chunks = []
current_chunk = []

for line in full_text.split("\n"):
    line = line.strip()  # remove extra spaces

    if line == "":  # skip empty lines
        continue

    # If line starts a new section (1. 2. 3. 4. 5.)
    if line[:2] in ["1.", "2.", "3.", "4.", "5."]:
        if current_chunk:  # save previous chunk if it exists
            chunks.append(" ".join(current_chunk))
        current_chunk = [line]  # start new chunk
    else:
        current_chunk.append(line)  # add line to current chunk

# Don't forget the last chunk
if current_chunk:
    chunks.append(" ".join(current_chunk))

# Print all chunks
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


--- Chunk 1 ---
SWIGGY DELIVERY PARTNER POLICY · 2024

--- Chunk 2 ---
1. WORKING HOURS Delivery partners can choose their own working hours. Minimum commitment is 4 hours per day to remain active on the platform. Partners working more than 8 hours get a productivity bonus of ·50 per day.

--- Chunk 3 ---
2. PAYMENT POLICY Base pay is ·25 per delivery. Long distance deliveries above 5km get an extra ·10. Payments are processed every Monday by 10am.

--- Chunk 4 ---
3. CANCELLATION POLICY Partners can cancel up to 2 orders per day without penalty. More than 2 cancellations results in a ·20 deduction per cancellation. Repeated cancellations may lead to account suspension.

--- Chunk 5 ---
4. RATINGS POLICY Partners must maintain a minimum rating of 3.5 out of 5. Partners below 3.5 rating for 2 consecutive weeks enter a performance review. Partners above 4.5 rating for a month receive a bonus of ·500.

--- Chunk 6 ---
5. SAFETY POLICY Helmets are mandatory at all times during delivery. V

## Step 5 — Send Relevant Chunk to LLM
A delivery partner has a question about cancellations.
We send only the relevant chunk to the LLM — not the entire document.
This saves tokens and gives a more focused answer.

In [6]:
# Set up Groq client
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# The delivery partner's question
question = "What happens if I cancel too many orders?"

# We already know Chunk 4 is relevant — index 3 in our list (0-based)
relevant_chunk = chunks[3]

print("Relevant chunk being sent to LLM:")
print(relevant_chunk)
print()

# Build the prompt
prompt = f"""You are a helpful assistant for Swiggy delivery partners.
Answer the partner's question using only the policy text provided below.
If the answer is not in the policy text, say "I don't know".

Policy text:
{relevant_chunk}

Partner's question: {question}

Answer in simple, friendly language.
"""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print("LLM Answer:")
print(response.choices[0].message.content)

Relevant chunk being sent to LLM:
3. CANCELLATION POLICY Partners can cancel up to 2 orders per day without penalty. More than 2 cancellations results in a ·20 deduction per cancellation. Repeated cancellations may lead to account suspension.

LLM Answer:
If you cancel more than 2 orders in a day, you'll get charged a penalty of ₹20 for each extra cancellation. And if you keep cancelling orders too often, your account might be suspended.


## What We Built Today
- Created a PDF document using PyMuPDF
- Extracted text from the PDF page by page
- Split the text into meaningful chunks by section
- Sent only the relevant chunk to the LLM
- Got a clean, friendly answer back

## AWS Equivalent
| What we did | AWS Service |
|---|---|
| Store PDF files | Amazon S3 |
| Extract text from PDF | Amazon Textract |
| Split into chunks | AWS Glue / Lambda |
| Find relevant chunk | Amazon OpenSearch Serverless |
| Send to LLM | Amazon Bedrock |

## What is Next
Session 3 — API Data Pipelines
We will pull live data from a real REST API, parse the JSON response,
and feed it into an LLM for analysis.